In [1]:
import numpy as np
import pandas as pd
import ast
import re
from sentence_transformers import SentenceTransformer
import torch

/Users/basmala.ayman/Documents/FCIS/GP/GitHub Repo/Hackathons-Team-Formation-pre-processing/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Members Embeddings

In [2]:
members = pd.read_csv('datasets/filtered_mapped_members.csv')

In [3]:
members.head(3)

,member_id,bio,hard_skills,hackathons_interests,projects_count,hackathons_count,achievements_count,followers_count,following_count,likes_count,winnings_count,Descriptions,hard_skills_clean,mapped_soft_skills
0,1,NaN,"['python', 'machine-learning', 'flask', 'djang...","['Machine Learning/AI', 'Productivity', 'Begin...",4,25,7,14,12,41,4,['I helped out with a mostly everything. I als...,"['flask', 'react', 'django', 'python', 'fastap...",[]
1,2,High school student with a passion for applyin...,"['python', 'javascript', 'java']","['Beginner Friendly', 'Low/No Code', 'Machine ...",1,3,5,2,2,1,0,NaN,"['java', 'javascript', 'python']",[]
2,3,Decentralized World,"['c', 'c++', 'golang', 'solidity', 'blockchain']",['Blockchain'],3,5,5,3,2,2,0,"[""I've been deeply involved as a Smart Contrac...","['golang', 'solidity', 'c', 'blockchain', 'c++']",[]


In [4]:
members.shape

(112454, 14)

In [5]:
members['hard_skills_clean'].apply(type).value_counts()

hard_skills_clean
<class 'str'>    112454
Name: count, dtype: int64

## Convert String Columns to Lists

In [6]:
def to_list(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    x = str(x).strip()
    if x == "" or x == "[]":
        return []
    if not (x.startswith("[") and x.endswith("]")):
        return [x]
    try:
        return ast.literal_eval(x)
    except:
        return []

In [7]:
members['hard_skills'] = members['hard_skills'].apply(to_list)
members['hackathons_interests'] = members['hackathons_interests'].apply(to_list)
members['Descriptions'] = members['Descriptions'].apply(to_list)
members['hard_skills_clean'] = members['hard_skills_clean'].apply(to_list)
members['mapped_soft_skills'] = members['mapped_soft_skills'].apply(to_list)

In [8]:
members['mapped_soft_skills'].apply(type).value_counts()

mapped_soft_skills
<class 'list'>    112454
Name: count, dtype: int64

In [9]:
members['bio'] = members['bio'].fillna('')

## Clean Text

In [10]:
def clean_text(text):
  text = str(text).lower()
  text = re.sub(r"http\S+", " ", text)
  text = re.sub(r"[_\-|,/]", " ", text)
  text = re.sub(r"\s+", " ", text).strip()
  return text

In [11]:
def clean_skill(token):
    token = str(token).lower().strip()
    token = re.sub(r"\s+", " ", token)
    token = re.sub(r"[^a-z0-9\+\#\.\- ]", " ", token) # keep + # . (for c++, c#, node.js)
    token = re.sub(r"\s+", " ", token).strip()
    return token

In [12]:
def uniq_clean_list(lst):
    if not isinstance(lst, list):
        return []
    out = []
    seen = set()
    for x in lst:
        x = clean_skill(x)
        if x and x not in seen:
            out.append(x)
            seen.add(x)
    return out

### Clean skills from duplications

In [13]:
members['hard_skills_clean'] = members['hard_skills_clean'].apply(uniq_clean_list)
members['mapped_soft_skills'] = members['mapped_soft_skills'].apply(uniq_clean_list)

## Build only one text field for each member

In [14]:
def build_member_text(member_row):
  bio = clean_text(member_row.get("bio", ""))
  hard = member_row.get('hard_skills_clean', [])
  soft = member_row.get('mapped_soft_skills', [])
  interests = member_row.get('hackathons_interests', [])
  desc_list = [clean_text(d) for d in member_row.get("Descriptions", [])]

  # convert lists into text
  hard_text = ', '.join(hard) if hard else ""
  soft_text = ', '.join(soft) if soft else ""
  interests_text = ", ".join(interests) if interests else ""
  desc_text = ' '.join(desc_list) if desc_list else ""

  parts = []
  if bio:
    parts.append(f"bio: {bio}")
  if hard_text:
    parts.append(f"hard skills: {hard_text}")
  if soft_text:
    parts.append(f"soft skills: {soft_text}")
  if interests_text:
    parts.append(f"hackathon interests: {interests_text}")
  if desc_text:
    parts.append(f"experience: {desc_text}")

  if not parts:
    return "no profile"

  return " | ".join(parts)

In [15]:
members['member_text'] = members.apply(build_member_text, axis=1)

In [16]:
members[['bio', 'hard_skills_clean', 'mapped_soft_skills', 'member_text']].head()

,bio,hard_skills_clean,mapped_soft_skills,member_text
0,,"[flask, react, django, python, fastapi, fireba...",[],"hard skills: flask, react, django, python, fas..."
1,High school student with a passion for applyin...,"[java, javascript, python]",[],bio: high school student with a passion for ap...
2,Decentralized World,"[golang, solidity, c, blockchain, c++]",[],bio: decentralized world | hard skills: golang...
3,"I am always open to collaboration, learning, a...","[java, flutter, android]",[],bio: i am always open to collaboration learnin...
4,,"[javascript, python]",[],"hard skills: javascript, python | hackathon in..."


## Generate Embeddings

In [17]:
# for macOs
device = "mps" if torch.backends.mps.is_available() else "cpu"
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
# for windows
# model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1942.18it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
members = members.reset_index(drop=True)
texts = members['member_text'].tolist()

In [19]:
embeddings = model.encode(
    texts,
    batch_size=128,
    show_progress_bar=True,
    normalize_embeddings=True # makes cosine similarity easier later
)

Batches: 100%|██████████| 879/879 [02:48<00:00,  5.22it/s]


## Save embeddings as npy file

In [20]:
np.save('embeddings/member_embeddings.npy', embeddings)
members[["member_id"]].to_csv('embeddings/member_ids.csv', index=False)

In [21]:
emb = np.load('embeddings/member_embeddings.npy')
ids = pd.read_csv('embeddings/member_ids.csv')

print(emb.shape)
print(len(ids))

(112454, 384)
112454


# Projects Embeddings

In [22]:
teams = pd.read_csv('datasets/filtered_teams_clean.csv')

In [23]:
teams.shape

(39269, 8)

In [24]:
teams.head(3)

,project_id,hackathon_id,members_count,is_winner,winners_description,project_tags,project_description,short_description
0,2501,14065,4,True,The Best Use of Assembly AI API...,"['assemblyai', 'dart', 'deep-translator', 'flu...",Inspiration The Babel Fish from Hitchhiker's G...,Hitchhiker's Guide to the Galaxy is a classic ...
1,592,14065,4,True,Best Hack | Runner Up 1\n ...,"['canva', 'dart', 'firebase', 'flutter', 'java...",💡 Inspiration 💡 We love sharing and writing st...,Bit-sized adventures right at your fingertips.
2,26991,14065,3,True,Jina AI - Best Hack Using Jina ...,"['assemblyai-api', 'docker', 'express.js', 'fi...",Inspiration 💡 The COVID-19 pandemic imposed se...,"Eat the Food, not the Time 👨‍🍳🍖"


In [25]:
teams = teams.drop(columns=['members_count','hackathon_id','is_winner', 'winners_description', 'project_description'])

In [26]:
teams.columns

Index(['project_id', 'project_tags', 'short_description'], dtype='str')

## Handel project tags

In [27]:
teams['project_tags'].apply(type).value_counts()

project_tags
<class 'str'>    39269
Name: count, dtype: int64

In [28]:
teams['project_tags']=teams['project_tags'].apply(to_list)

In [29]:
teams['project_tags'].apply(type).value_counts()

project_tags
<class 'list'>    39269
Name: count, dtype: int64

### Clean project tags from duplications

In [30]:
teams['project_tags']=teams['project_tags'].apply(uniq_clean_list)

### Handle nulls in short_description

In [31]:
teams["short_description"] = teams["short_description"].fillna("")

## Build one text field for each project

In [32]:
def build_project_text(team_row):

  tags = team_row.get('project_tags', [])
  desc = clean_text(team_row.get("short_description", ""))

  tags_text = ', '.join(tags) if tags else ""

  parts = []
  if desc:
      parts.append(f"title: {desc}")
  if tags_text:
      parts.append(f"project was built with: {tags_text}")

  return " | ".join(parts)

In [33]:
teams["project_text"] = teams.apply(build_project_text, axis=1)

In [34]:
teams.head(3)

,project_id,project_tags,short_description,project_text
0,2501,"[assemblyai, dart, deep-translator, flutter, i...",Hitchhiker's Guide to the Galaxy is a classic ...,title: hitchhiker's guide to the galaxy is a c...
1,592,"[canva, dart, firebase, flutter, java, numpy, ...",Bit-sized adventures right at your fingertips.,title: bit sized adventures right at your fing...
2,26991,"[assemblyai-api, docker, express.js, firebase,...","Eat the Food, not the Time 👨‍🍳🍖",title: eat the food not the time 👨‍🍳🍖 | projec...


## Build Embeddings

In [35]:
teams = teams.reset_index(drop=True)
project_texts = teams["project_text"].tolist()

In [36]:
project_embeddings = model.encode(
    project_texts,
    batch_size=128,
    show_progress_bar=True,
    normalize_embeddings=True
)

Batches: 100%|██████████| 307/307 [01:28<00:00,  3.47it/s]


## Save Result

In [37]:
np.save(
    "embeddings/project_embeddings.npy",
    project_embeddings
)

teams[["project_id"]].to_csv(
    "embeddings/project_ids.csv",
    index=False
)

In [38]:
pro_emb = np.load('embeddings/project_embeddings.npy')
pro_ids = pd.read_csv('embeddings/project_ids.csv')

print(pro_emb.shape)
print(len(pro_ids))

(39269, 384)
39269


# Check Embeddings

In [39]:
assert len(members) == embeddings.shape[0]
assert len(teams) == project_embeddings.shape[0]

print(members["member_text"].head(3).tolist())
print(teams["project_text"].head(3).tolist())

['hard skills: flask, react, django, python, fastapi, firebase, machine-learning | hackathon interests: Machine Learning/AI, Productivity, Beginner Friendly, Social Good, Education, Low/No Code | experience: i helped out with a mostly everything. i also worked on the submission. i worked on the majority of the backend and the design of the ner informed recommendation engine. i also worked on the devpost submission. i worked on the django backend and the submission and presentation. i made the city searching algorithm and helped with the flask backend. i worked on the written question checker the multiple choice checker and did most of the streamlit code', 'bio: high school student with a passion for applying computer science to solve scientific problems | hard skills: java, javascript, python | hackathon interests: Beginner Friendly, Low/No Code, Machine Learning/AI', "bio: decentralized world | hard skills: golang, solidity, c, blockchain, c++ | hackathon interests: Blockchain | exper

In [40]:
print("Empty member_text:", (members["member_text"].str.strip() == "").sum())
print("Empty project_text:", (teams["project_text"].str.strip() == "").sum())

Empty member_text: 0
Empty project_text: 0
